# 04 — Correlación: Brecha Salarial Tech ↔ Precios de Vivienda ZMG

**Proyecto:** GDL Ecosystem Intelligence  
**Fuentes:** ENOE (INEGI) + INFONAVIT / SHF — Índice de Precios de Vivienda  

---
**Hipótesis:** El incremento de salarios en el sector tech, impulsado por el nearshoring,  
está correlacionado con el aumento en los precios de vivienda en la ZMG.

In [ ]:
# ── 0. Entorno ────────────────────────────────────────────────────────────────
import os
import sys
from pathlib import Path

project_root = Path(r'C:\Users\emmys\OneDrive\Documents\GDL-ECO-INT')
os.chdir(project_root)
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

print(f'✅ Directorio: {os.getcwd()}')

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr

plt.rcParams.update({
    'figure.dpi': 130,
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor': '#FAFAFA',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
PALETTE_TECH    = '#1A73E8'
PALETTE_HOUSING = '#F9A825'
PALETTE_NEU     = '#5F6368'
PALETTE_CORR    = '#34A853'

print('✅ Librerías cargadas')

In [ ]:
# ── 2. Cargar brecha salarial (output del notebook 01) ───────────────────────
enoe_path = Path('data/processed/enoe_brecha_salarial_zmg.csv')
df_enoe = pd.read_csv(enoe_path)

print(df_enoe.columns.tolist())
print(df_enoe.head(2))

# Calcular mediana tech y no-tech por trimestre (ZMG agregado)
enoe_pivot = (
    df_enoe
    .groupby(['periodo', 'sector'])['mediana']
    .mean()
    .reset_index()
    .pivot(index='periodo', columns='sector', values='mediana')
    .reset_index()
    .rename(columns={'periodo': 'trimestre', 'Tech': 'mediana_tech', 'No-Tech': 'mediana_notech'})
)
enoe_pivot.columns.name = None

enoe_pivot['brecha_pct'] = (
    (enoe_pivot['mediana_tech'] / enoe_pivot['mediana_notech'] - 1) * 100
).round(1)

print(f'✅ ENOE cargado: {len(enoe_pivot)} trimestres')
print(enoe_pivot[['trimestre', 'mediana_tech', 'mediana_notech', 'brecha_pct']].head())

In [ ]:
# ── 3. Cargar índice SHF de precios de vivienda ───────────────────────────────
vivienda_path = Path('data/raw/vivienda/shf_indice_zmg.csv')

df_viv = pd.read_csv(vivienda_path)
df_viv = df_viv[['trimestre', 'indice_shf', 'variacion_anual_pct']].copy()

print(f'✅ SHF cargado: {len(df_viv)} trimestres')
print(f'   Rango: {df_viv["trimestre"].min()} — {df_viv["trimestre"].max()}')
display(df_viv)

In [ ]:
# ── 4. Merge: ENOE + Vivienda por trimestre ───────────────────────────────────
enoe_path = Path('data/processed/enoe_brecha_salarial_zmg.csv')
df_enoe = pd.read_csv(enoe_path)

# Pivotar: una fila por periodo, columnas Tech / No-Tech con mediana
enoe_pivot = (
    df_enoe
    .pivot_table(index='periodo', columns='sector', values='mediana', aggfunc='mean')
    .reset_index()
)
enoe_pivot.columns.name = None
enoe_pivot = enoe_pivot.rename(columns={
    'periodo':  'trimestre',
    'Tech':     'mediana_tech',
    'No-Tech':  'mediana_notech',
})
enoe_pivot['brecha_pct'] = (
    (enoe_pivot['mediana_tech'] / enoe_pivot['mediana_notech'] - 1) * 100
).round(1)

# Inner join — solo trimestres presentes en ambas fuentes
df_merged = enoe_pivot.merge(df_viv, on='trimestre', how='inner')

# Variaciones trimestrales
df_merged['delta_mediana_tech_pct'] = df_merged['mediana_tech'].pct_change() * 100
df_merged['delta_vivienda_pct']     = df_merged['indice_shf'].pct_change() * 100

print(f'ENOE disponible : {len(enoe_pivot)} trimestres')
print(f'SHF disponible  : {len(df_viv)} trimestres')
print(f'Fusionado       : {len(df_merged)} trimestres en común')
print()
print(df_merged[['trimestre', 'mediana_tech', 'mediana_notech', 'brecha_pct',
                 'indice_shf', 'variacion_anual_pct']].to_string(index=False))

In [ ]:
# ── Diagnóstico: conteo de registros tech por trimestre ───────────────────────
n_tech = (
    df_enoe[df_enoe['sector'] == 'Tech'][['periodo', 'n']]
    .rename(columns={'periodo': 'trimestre', 'n': 'n_tech'})
)

diag = df_merged[['trimestre', 'mediana_tech', 'brecha_pct']].merge(n_tech, on='trimestre', how='left')
diag['n_tech'] = pd.to_numeric(diag['n_tech'], errors='coerce')
diag['excluir'] = diag['n_tech'] < 50

print(f'{"trimestre":<12} {"n_tech":>8} {"mediana_tech":>14} {"brecha_pct":>12}  {"⚠️ excluir?"}')
print('-' * 60)
for _, row in diag.iterrows():
    flag = '  ⚠️  < 50' if row['excluir'] else ''
    print(f'{row["trimestre"]:<12} {int(row["n_tech"]) if pd.notna(row["n_tech"]) else "N/A":>8}'
          f' {row["mediana_tech"]:>14.2f} {row["brecha_pct"]:>11.1f}%{flag}')

excluidos = diag[diag['excluir']]['trimestre'].tolist()
print(f'\nTrimestres con n_tech < 50: {excluidos if excluidos else "ninguno"}')

In [ ]:
# ── 5. VIZ 8: Doble eje — salario tech & índice SHF vivienda ──────────────────
fig, ax1 = plt.subplots(figsize=(13, 6), facecolor='#FAFAFA')
ax2 = ax1.twinx()

x = range(len(df_merged))
xticks = df_merged['trimestre'].tolist()

# Eje izquierdo: mediana salarial tech
l1 = ax1.plot(x, df_merged['mediana_tech'], color=PALETTE_TECH,
              linewidth=2.5, marker='o', markersize=7, label='Mediana salarial Tech')
ax1.fill_between(x, df_merged['mediana_notech'], df_merged['mediana_tech'],
                 alpha=0.1, color=PALETTE_TECH, label='Brecha salarial')
l2 = ax1.plot(x, df_merged['mediana_notech'], color='#EA4335',
              linewidth=1.5, marker='s', markersize=5, linestyle='--', label='Mediana No-Tech')

# Eje derecho: índice SHF
l3 = ax2.plot(x, df_merged['indice_shf'], color=PALETTE_HOUSING,
              linewidth=2.5, marker='^', markersize=7, label='Índice SHF vivienda')

ax1.set_xticks(x)
ax1.set_xticklabels(xticks, rotation=45, ha='right')
ax1.set_ylabel('Ingreso mediano por hora (MXN)', color=PALETTE_TECH)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
ax2.set_ylabel('Índice SHF (base 2017=100)', color=PALETTE_HOUSING)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.1f}'))

ax1.set_title('Salarios Tech vs. Índice SHF de Vivienda — ZMG\n¿Existe correlación con el boom nearshoring?',
              fontweight='bold', pad=15)

lines = l1 + l2 + l3
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left', frameon=False, fontsize=9)

plt.tight_layout()
plt.savefig('reports/figures/08_salario_tech_vs_vivienda.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Figura guardada → reports/figures/08_salario_tech_vs_vivienda.png')

In [ ]:
# ── 6. VIZ 9: Scatter + regresión brecha salarial vs índice SHF ───────────────
df_corr = df_merged[['brecha_pct', 'indice_shf']].dropna()

pearson_r, pearson_p = pearsonr(df_corr['brecha_pct'], df_corr['indice_shf'])
spearman_r, spearman_p = spearmanr(df_corr['brecha_pct'], df_corr['indice_shf'])

fig, ax = plt.subplots(figsize=(9, 7), facecolor='#FAFAFA')

scatter = ax.scatter(df_corr['brecha_pct'], df_corr['indice_shf'],
                     c=range(len(df_corr)), cmap='Blues', s=100, zorder=5,
                     edgecolors='white', linewidths=1.5)

for i, (_, row) in enumerate(df_merged[['trimestre', 'brecha_pct', 'indice_shf']].dropna().iterrows()):
    ax.annotate(row['trimestre'], (row['brecha_pct'], row['indice_shf']),
                xytext=(5, 5), textcoords='offset points', fontsize=8, color=PALETTE_NEU)

if len(df_corr) > 2:
    m, b = np.polyfit(df_corr['brecha_pct'], df_corr['indice_shf'], 1)
    xfit = np.linspace(df_corr['brecha_pct'].min(), df_corr['brecha_pct'].max(), 100)
    ax.plot(xfit, m * xfit + b, color=PALETTE_CORR, linewidth=2,
            linestyle='--', alpha=0.8, label=f'Regresión lineal (r={pearson_r:.2f})')

ax.set_title('Correlación: Brecha Salarial Tech ↔ Índice SHF Vivienda\nZMG',
             fontweight='bold', pad=15)
ax.set_xlabel('Brecha salarial Tech vs No-Tech (%)')
ax.set_ylabel('Índice SHF (base 2017=100)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.1f}'))
ax.legend(frameon=False)

stats_text = (
    f'Pearson r = {pearson_r:.3f}  (p={pearson_p:.3f})\n'
    f'Spearman ρ = {spearman_r:.3f}  (p={spearman_p:.3f})'
)
ax.text(0.97, 0.05, stats_text, transform=ax.transAxes, ha='right',
        va='bottom', fontsize=9, color=PALETTE_NEU,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig('reports/figures/09_scatter_brecha_vivienda.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Figura guardada → reports/figures/09_scatter_brecha_vivienda.png')

In [ ]:
# ── 8. Exportar dataset fusionado ─────────────────────────────────────────────
export_path = Path('data/processed/correlacion_salario_vivienda.csv')
df_merged.to_csv(export_path, index=False, encoding='utf-8-sig')
print(f'✅ CSV exportado → {export_path}')

# ── 9. Hallazgos clave ────────────────────────────────────────────────────────
print()
print('=' * 60)
print('📌 HALLAZGOS CLAVE — CORRELACIÓN SALARIAL / VIVIENDA')
print('=' * 60)
print(f'  Pearson r  = {pearson_r:.3f}  (p={pearson_p:.4f})')
print(f'  Spearman ρ = {spearman_r:.3f}  (p={spearman_p:.4f})')
sig    = 'SÍ' if pearson_p < 0.05 else 'NO'
direc  = 'POSITIVA' if pearson_r > 0 else 'NEGATIVA'
intens = (
    'FUERTE'   if abs(pearson_r) > 0.7 else
    'MODERADA' if abs(pearson_r) > 0.4 else 'DÉBIL'
)
print(f'  Resultado: Correlación {direc} {intens} — Significativa: {sig}')
print(f'  Interpretación: Por cada +1pp de brecha salarial tech,')
indice_delta = df_corr['indice_shf'].std() / df_corr['brecha_pct'].std() * pearson_r
print(f'  el índice SHF varía ~{indice_delta:.2f} puntos en promedio.')
print('=' * 60)
print()
print('⚠️  NOTA METODOLÓGICA: Correlación ≠ causalidad.')
print('  El análisis no controla por otros factores (IED, tasas, oferta).')
print('  Algunos valores del índice SHF son interpolados — ver docs/fuentes.md.')
print()
print('🔴 LIMITACIÓN: Todos los trimestres tienen n_tech < 50 en la muestra')
print('  ENOE-ZMG. Con muestras pequeñas la mediana es inestable — los valores')
print('  extremos de 2025-T1 (87.5%) y 2025-T2 (3.6%) son ruido estadístico,')
print('  no tendencia real. La correlación r=0.001 no es interpretable con este')
print('  tamaño muestral. Recomendación: complementar con registros IMSS para')
print('  obtener el universo completo de trabajadores tech formales en ZMG.')

In [ ]:
# ── 8. Exportar dataset fusionado ─────────────────────────────────────────────
export_path = Path('data/processed/correlacion_salario_vivienda.csv')
df_merged.to_csv(export_path, index=False, encoding='utf-8-sig')
print(f'✅ CSV exportado → {export_path}')

# ── 9. Hallazgos clave ────────────────────────────────────────────────────────
print()
print('=' * 60)
print('📌 HALLAZGOS CLAVE — CORRELACIÓN SALARIAL / VIVIENDA')
print('=' * 60)
print(f'  Pearson r  = {pearson_r:.3f}  (p={pearson_p:.4f})')
print(f'  Spearman ρ = {spearman_r:.3f}  (p={spearman_p:.4f})')
sig    = 'SÍ' if pearson_p < 0.05 else 'NO'
direc  = 'POSITIVA' if pearson_r > 0 else 'NEGATIVA'
intens = (
    'FUERTE'   if abs(pearson_r) > 0.7 else
    'MODERADA' if abs(pearson_r) > 0.4 else 'DÉBIL'
)
print(f'  Resultado: Correlación {direc} {intens} — Significativa: {sig}')
print(f'  Interpretación: Por cada +1pp de brecha salarial tech,')
indice_delta = df_corr['indice_shf'].std() / df_corr['brecha_pct'].std() * pearson_r
print(f'  el índice SHF varía ~{indice_delta:.2f} puntos en promedio.')
print('=' * 60)
print('\n⚠️  NOTA METODOLÓGICA: Correlación ≠ causalidad.')
print('  El análisis no controla por otros factores (IED, tasas, oferta).')
print('  Algunos valores del índice SHF son interpolados — ver docs/fuentes.md.')

## Próximos pasos — Fuente alternativa

La muestra ENOE-ZMG para sector tech es insuficiente para análisis de correlación robusto.
Fuente recomendada para el siguiente paso:

**IMSS — Registros patronales por municipio y actividad económica**
- Cobertura: universo de trabajadores formales (no muestra)
- Desagregación: municipio × rama SCIAN × trimestre
- URL: https://www.imss.gob.mx/estadisticas/financieras/index.htm

Con datos IMSS se puede obtener el conteo real de asegurados en SCIAN 51 por municipio ZMG,
eliminar la dependencia de la expansión muestral de ENOE, y repetir el análisis de correlación
con un N suficiente para que los estadísticos sean interpretables.

---
## Próximos pasos
- **Semana 3:** Dashboard Power BI con los 3 CSVs exportados
- **Pendiente:** Reemplazar datos de vivienda de referencia con datos reales SHF

**Commit sugerido:**
```bash
git add notebooks/04_correlacion_vivienda.ipynb data/processed/correlacion_salario_vivienda.csv reports/figures/08*.png reports/figures/09*.png reports/figures/10*.png
git commit -m "analysis: correlación brecha salarial tech vs precios vivienda ZMG"
```